In [0]:
import pandas as pd
import numpy as np

# don't display print or show large data - slow 
# use .limit() to explore data (faster and safer than using the full table)
# filter early 
# convert to Pandas only on smaller results 
# reuse data frames (refrence already loaded data instead of loading over and over)

# spark = getting the data 
# pandas = working with the data 
hno = spark.table("workspace.unochadatasets.hpc_hno_2025")
fts = spark.table("workspace.unochadatasets.fts_requirements_funding_global")

# Get parameter values
crisis_category = dbutils.widgets.get("crisis_category")
include_temporal = dbutils.widgets.get("include_temporal_factor")

# HNO baseline - now filtered by selected crisis category
hno_base_spark = spark.sql(f"""
SELECT
  `Country ISO3` AS iso3,
  `In Need` AS people_in_need
FROM workspace.unochadatasets.hpc_hno_2025
WHERE `Cluster` = '{crisis_category}'
  AND (`Category` IS NULL OR TRIM(`Category`) = '')
""")

hno_base = hno_base_spark.toPandas()
hno_base["people_in_need"] = pd.to_numeric(hno_base["people_in_need"], errors="coerce")
hno_base = hno_base.dropna(subset=["iso3", "people_in_need"])

# FTS baseline - Filter to 2025 to match HNO data
# requirements = how much funding is needed
# funding = how much funding is actually provided
fts_base_spark = spark.sql("""
SELECT
  `countryCode` AS iso3,
  SUM(`requirements`) AS requirements,
  SUM(`funding`) AS funding
FROM workspace.unochadatasets.fts_requirements_funding_global
WHERE `year` = 2025
GROUP BY `countryCode`
""")

fts_base = fts_base_spark.toPandas()
fts_base["requirements"] = pd.to_numeric(fts_base["requirements"], errors="coerce")
fts_base["funding"] = pd.to_numeric(fts_base["funding"], errors="coerce")
fts_base = fts_base.dropna(subset=["iso3", "requirements", "funding"])

# Temporal factor calculation (only if enabled)
if include_temporal == "Yes":
    # Get historical funding data (2020-2024) to calculate temporal factor
    fts_historical_spark = spark.sql("""
    SELECT
      `countryCode` AS iso3,
      `year`,
      SUM(`requirements`) AS requirements,
      SUM(`funding`) AS funding
    FROM workspace.unochadatasets.fts_requirements_funding_global
    WHERE `year` BETWEEN 2020 AND 2024
    GROUP BY `countryCode`, `year`
    """)
    
    fts_historical = fts_historical_spark.toPandas()
    fts_historical["requirements"] = pd.to_numeric(fts_historical["requirements"], errors="coerce")
    fts_historical["funding"] = pd.to_numeric(fts_historical["funding"], errors="coerce")
    fts_historical = fts_historical.dropna(subset=["iso3", "requirements", "funding"])
    fts_historical = fts_historical[fts_historical["requirements"] > 0].copy()
    
    # Calculate historical coverage ratios
    fts_historical["coverage_ratio"] = (fts_historical["funding"] / fts_historical["requirements"]).clip(lower=0, upper=1)
    
    # NEW APPROACH: Calculate actual gap scores for each historical year
    # For historical years, we approximate people_in_need from requirements (as we don't have historical HNO data)
    # gap_score_year = requirements × (1 - coverage_ratio)
    fts_historical["gap_score_year"] = fts_historical["requirements"] * (1 - fts_historical["coverage_ratio"])
    
    # Sum historical gaps per country
    historical_gaps = fts_historical.groupby("iso3")["gap_score_year"].sum().reset_index()
    historical_gaps.columns = ["iso3", "cumulative_historical_gap"]
    
    # Count years for reference
    years_count = fts_historical.groupby("iso3")["year"].count().reset_index()
    years_count.columns = ["iso3", "years_with_data"]
    
    temporal_factor = historical_gaps.merge(years_count, on="iso3", how="left")
else:
    # Create empty temporal factor
    temporal_factor = pd.DataFrame(columns=["iso3", "cumulative_historical_gap", "years_with_data"])

# Merge - based on country 
df = hno_base.merge(fts_base, on="iso3", how="inner")

# Merge temporal factor
df = df.merge(temporal_factor, on="iso3", how="left")
df["cumulative_historical_gap"] = df["cumulative_historical_gap"].fillna(0).astype('float64')
df["years_with_data"] = df["years_with_data"].fillna(0).astype('int64')

# Filter - remove rows where some data is missing 
df = df[(df["people_in_need"] > 0) & (df["requirements"] > 0)].copy()

# Already one row per country from the aggregation in SQL query above
df_country = df.copy()

# Compute baseline
df_country["coverage_ratio"] = df_country["funding"] / df_country["requirements"] # how much of the funding was actually provided 
df_country["coverage_ratio"] = df_country["coverage_ratio"].clip(lower=0, upper=1) 

# Base gap score: how many people need help and how underfunded it is
df_country["base_gap_score"] = df_country["people_in_need"] * (1 - df_country["coverage_ratio"])

# Apply temporal multiplier based on parameter setting
if include_temporal == "Yes":
    # NEW FORMULA: Use cumulative historical gaps to calculate multiplier
    # The more unmet need accumulated over years, the higher the multiplier
    # temporal_multiplier = 1 + (average_annual_historical_gap / current_base_gap_score) * weight
    
    # Calculate average annual historical gap
    df_country["avg_annual_historical_gap"] = df_country["cumulative_historical_gap"] / df_country["years_with_data"].replace(0, 1)
    
    # Temporal weight determines how much history influences current score
    TEMPORAL_WEIGHT = 0.5  # 50% weight means if avg historical gap = current gap, you get 1.5x multiplier
    
    # Calculate multiplier: larger historical gaps relative to current gap = higher multiplier
    df_country["temporal_multiplier"] = 1 + (df_country["avg_annual_historical_gap"] / df_country["base_gap_score"].replace(0, 1)) * TEMPORAL_WEIGHT
    
    # Cap the multiplier to avoid extreme values
    df_country["temporal_multiplier"] = df_country["temporal_multiplier"].clip(lower=1.0, upper=3.0)
else:
    # No temporal adjustment - all countries get multiplier of 1.0
    df_country["temporal_multiplier"] = 1.0
    df_country["avg_annual_historical_gap"] = 0.0

# Final gap score with temporal factor (if enabled)
df_country["gap_score"] = df_country["base_gap_score"] * df_country["temporal_multiplier"]

# Rank
ranked = df_country.sort_values("gap_score", ascending=False).reset_index(drop=True)

final = ranked[[
    "iso3",
    "people_in_need",
    "requirements",
    "funding",
    "coverage_ratio",
    "cumulative_historical_gap",
    "years_with_data",
    "temporal_multiplier",
    "gap_score"
]].copy()

# Ensure consistent data types
final["people_in_need"] = final["people_in_need"].astype('int64')
final["cumulative_historical_gap"] = final["cumulative_historical_gap"].astype('float64')
final["years_with_data"] = final["years_with_data"].astype('int64')

# Convert pandas DataFrame back to Spark and save as a table
# Use overwriteSchema to handle schema changes without needing DROP permissions
final_spark = spark.createDataFrame(final)
final_spark.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.unochadatasets.gap_rankings_2025")

print(f"Data saved to workspace.unochadatasets.gap_rankings_2025")
print(f"  - Crisis Category: {crisis_category}")
print(f"  - Temporal Factor: {include_temporal}")
print(f"  - Total countries ranked: {len(final)}")

if include_temporal == "Yes":
    print(f"\nTemporal Factor Stats (based on historical gap scores):")
    print(f"  - Average cumulative historical gap: ${final['cumulative_historical_gap'].mean()/1e9:.2f}B")
    print(f"  - Max cumulative historical gap: ${final['cumulative_historical_gap'].max()/1e9:.2f}B ({final.loc[final['cumulative_historical_gap'].idxmax(), 'iso3']})")
    print(f"  - Average temporal multiplier: {final['temporal_multiplier'].mean():.2f}x")
    print(f"  - Max temporal multiplier: {final['temporal_multiplier'].max():.2f}x")
    print(f"  - Countries with multiplier > 2.0x: {len(final[final['temporal_multiplier'] > 2.0])}")
else:
    print(f"\nTemporal factor disabled - rankings based only on 2025 data")

In [0]:
# Export gap rankings for frontend development
# Write to Unity Catalog Volume (serverless-compatible)

# Use the exports volume
volume_path = "/Volumes/workspace/unochadatasets/exports"

# Export as CSV
csv_path = f"{volume_path}/gap_rankings_2025.csv"
final.to_csv(csv_path, index=False)
print(f"✓ CSV exported to: {csv_path}")

# Export as JSON
json_path = f"{volume_path}/gap_rankings_2025.json"
final.to_json(json_path, orient="records", indent=2)
print(f"✓ JSON exported to: {json_path}")

print(f"\nTo download these files:")
print(f"1. Navigate to Catalog Explorer")
print(f"2. Go to workspace > unochadatasets > exports volume")
print(f"3. Right-click the file and select 'Download'")

In [0]:
# Display top 20 countries by gap score
display(final.head(20))

In [0]:
import os
import pandas as pd
import numpy as np
from datetime import datetime

# All parameter combinations
crisis_categories = ['ALL', 'CCM', 'CSS', 'EDU', 'ERY', 'FSC', 'HEA', 'LOG', 'MPC', 'MS', 'NUT', 'PRO', 'PRO-CPN', 'PRO-GBV', 'PRO-HLP', 'PRO-MIN', 'SHL', 'TEL', 'WSH']
temporal_options = ['Yes', 'No']

repo_path = "/Workspace/Repos/anicernak@gmail.com/UN-OCHA-/data/rankings"
os.makedirs(repo_path, exist_ok=True)

# Load lookup table for country names
country_lookup = spark.table("workspace.unochadatasets.country_iso3_lookup").toPandas()

print(f"Generating {len(crisis_categories) * len(temporal_options)} ranking combinations...\n")

generation_log = []
start_time = datetime.now()

for crisis_cat in crisis_categories:
    for temporal in temporal_options:
        try:
            # HNO baseline - filtered by crisis category
            hno_base_spark = spark.sql(f"""
            SELECT
              `Country ISO3` AS iso3,
              `In Need` AS people_in_need
            FROM workspace.unochadatasets.hpc_hno_2025
            WHERE `Cluster` = '{crisis_cat}'
              AND (`Category` IS NULL OR TRIM(`Category`) = '')
            """)
            
            hno_base = hno_base_spark.toPandas()
            hno_base["people_in_need"] = pd.to_numeric(hno_base["people_in_need"], errors="coerce")
            hno_base = hno_base.dropna(subset=["iso3", "people_in_need"])
            
            # FTS baseline
            fts_base_spark = spark.sql("""
            SELECT
              `countryCode` AS iso3,
              SUM(`requirements`) AS requirements,
              SUM(`funding`) AS funding
            FROM workspace.unochadatasets.fts_requirements_funding_global
            WHERE `year` = 2025
            GROUP BY `countryCode`
            """)
            
            fts_base = fts_base_spark.toPandas()
            fts_base["requirements"] = pd.to_numeric(fts_base["requirements"], errors="coerce")
            fts_base["funding"] = pd.to_numeric(fts_base["funding"], errors="coerce")
            fts_base = fts_base.dropna(subset=["iso3", "requirements", "funding"])
            
            # Temporal factor calculation
            if temporal == "Yes":
                fts_historical_spark = spark.sql("""
                SELECT
                  `countryCode` AS iso3,
                  `year`,
                  SUM(`requirements`) AS requirements,
                  SUM(`funding`) AS funding
                FROM workspace.unochadatasets.fts_requirements_funding_global
                WHERE `year` BETWEEN 2020 AND 2024
                GROUP BY `countryCode`, `year`
                """)
                
                fts_historical = fts_historical_spark.toPandas()
                fts_historical["requirements"] = pd.to_numeric(fts_historical["requirements"], errors="coerce")
                fts_historical["funding"] = pd.to_numeric(fts_historical["funding"], errors="coerce")
                fts_historical = fts_historical.dropna(subset=["iso3", "requirements", "funding"])
                fts_historical = fts_historical[fts_historical["requirements"] > 0].copy()
                
                fts_historical["coverage_ratio"] = (fts_historical["funding"] / fts_historical["requirements"]).clip(lower=0, upper=1)
                fts_historical["gap_score_year"] = fts_historical["requirements"] * (1 - fts_historical["coverage_ratio"])
                
                historical_gaps = fts_historical.groupby("iso3")["gap_score_year"].sum().reset_index()
                historical_gaps.columns = ["iso3", "cumulative_historical_gap"]
                
                years_count = fts_historical.groupby("iso3")["year"].count().reset_index()
                years_count.columns = ["iso3", "years_with_data"]
                
                temporal_factor = historical_gaps.merge(years_count, on="iso3", how="left")
            else:
                temporal_factor = pd.DataFrame(columns=["iso3", "cumulative_historical_gap", "years_with_data"])
            
            # Merge all data
            df = hno_base.merge(fts_base, on="iso3", how="inner")
            df = df.merge(temporal_factor, on="iso3", how="left")
            df["cumulative_historical_gap"] = df["cumulative_historical_gap"].fillna(0).astype('float64')
            df["years_with_data"] = df["years_with_data"].fillna(0).astype('int64')
            
            df = df[(df["people_in_need"] > 0) & (df["requirements"] > 0)].copy()
            
            # Calculate gap scores
            df["coverage_ratio"] = (df["funding"] / df["requirements"]).clip(lower=0, upper=1)
            df["base_gap_score"] = df["people_in_need"] * (1 - df["coverage_ratio"])
            
            if temporal == "Yes":
                df["avg_annual_historical_gap"] = df["cumulative_historical_gap"] / df["years_with_data"].replace(0, 1)
                TEMPORAL_WEIGHT = 0.5
                df["temporal_multiplier"] = 1 + (df["avg_annual_historical_gap"] / df["base_gap_score"].replace(0, 1)) * TEMPORAL_WEIGHT
                df["temporal_multiplier"] = df["temporal_multiplier"].clip(lower=1.0, upper=3.0)
            else:
                df["temporal_multiplier"] = 1.0
                df["avg_annual_historical_gap"] = 0.0
            
            df["gap_score"] = df["base_gap_score"] * df["temporal_multiplier"]
            
            # Rank
            ranked = df.sort_values("gap_score", ascending=False).reset_index(drop=True)
            
            final = ranked[[
                "iso3",
                "people_in_need",
                "requirements",
                "funding",
                "coverage_ratio",
                "cumulative_historical_gap",
                "years_with_data",
                "temporal_multiplier",
                "gap_score"
            ]].copy()
            
            # Add country names
            final = final.merge(country_lookup, on="iso3", how="left")
            
            # Reorder columns to put country_name after iso3
            cols = ['iso3', 'country_name'] + [col for col in final.columns if col not in ['iso3', 'country_name']]
            final = final[cols]
            
            # Generate filename
            temporal_suffix = "with_temporal" if temporal == "Yes" else "no_temporal"
            filename_base = f"gap_rankings_{crisis_cat.lower()}_{temporal_suffix}"
            
            # Export CSV
            csv_path = f"{repo_path}/{filename_base}.csv"
            final.to_csv(csv_path, index=False)
            
            # Export JSON
            json_path = f"{repo_path}/{filename_base}.json"
            final.to_json(json_path, orient="records", indent=2)
            
            generation_log.append({
                "crisis_category": crisis_cat,
                "temporal_factor": temporal,
                "countries": len(final),
                "status": "success"
            })
            
            print(f"✓ {crisis_cat} ({temporal}): {len(final)} countries")
            
        except Exception as e:
            generation_log.append({
                "crisis_category": crisis_cat,
                "temporal_factor": temporal,
                "countries": 0,
                "status": f"error: {str(e)}"
            })
            print(f"✗ {crisis_cat} ({temporal}): Error - {str(e)}")

end_time = datetime.now()
duration = (end_time - start_time).total_seconds()

print(f"\n{'='*60}")
print(f"Generation complete in {duration:.1f} seconds")
print(f"Files saved to: {repo_path}")
print(f"\nSummary:")

log_df = pd.DataFrame(generation_log)
successful = len(log_df[log_df['status'] == 'success'])
print(f"  - Successful: {successful}/{len(log_df)}")
print(f"  - Total countries across all combinations: {log_df[log_df['status'] == 'success']['countries'].sum()}")

if successful < len(log_df):
    print(f"\nFailed combinations:")
    failed = log_df[log_df['status'] != 'success']
    for _, row in failed.iterrows():
        print(f"  - {row['crisis_category']} ({row['temporal_factor']}): {row['status']}")